# 01 — Análisis Exploratorio de Datos (EDA)
## Aircraft Engine Predictive Maintenance · NASA C-MAPSS Dataset

**Objetivo de este notebook:** entender los datos antes de tocarlos.
Antes de entrenar cualquier modelo, necesitamos saber:
- ¿Cómo está estructurado el dataset?
- ¿Qué sensores son útiles y cuáles no?
- ¿Cómo se degrada un motor a lo largo del tiempo?
- ¿Qué variable vamos a predecir?

---

## 1. Importar librerías

Cargamos las herramientas que vamos a usar en este notebook.

In [ ]:
import pandas as pd        # Para manejar tablas de datos
import numpy as np         # Para cálculos numéricos
import matplotlib.pyplot as plt  # Para hacer gráficos
import seaborn as sns      # Para gráficos más bonitos
import warnings
warnings.filterwarnings('ignore')

# Estilo de los gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Librerías cargadas correctamente ✅')

## 2. Cargar los datos

El dataset NASA C-MAPSS tiene 4 subconjuntos (FD001 a FD004).
Cada uno simula condiciones operativas distintas.
Empezamos con FD001, el más sencillo (1 condición operativa, 1 tipo de fallo).

In [ ]:
# Nombres de las columnas del dataset (el archivo no tiene cabecera)
columnas = [
    'motor_id',      # Número identificador del motor
    'ciclo',         # Ciclo operacional (equivale a un vuelo)
    'setting_1',     # Condición operativa 1
    'setting_2',     # Condición operativa 2  
    'setting_3',     # Condición operativa 3
    's1', 's2', 's3', 's4', 's5',    # Lecturas de sensores 1-5
    's6', 's7', 's8', 's9', 's10',   # Lecturas de sensores 6-10
    's11', 's12', 's13', 's14', 's15', # Lecturas de sensores 11-15
    's16', 's17', 's18', 's19', 's20', 's21'  # Lecturas de sensores 16-21
]

# Cargamos el archivo de entrenamiento
df = pd.read_csv(
    '../data/raw/train_FD001.txt',
    sep=' ',           # Los valores están separados por espacios
    header=None,       # No hay fila de cabecera en el archivo
    names=columnas,    # Asignamos nuestros nombres
    index_col=False
)

# El archivo tiene 2 columnas vacías al final, las eliminamos
df = df.dropna(axis=1, how='all')

print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(10)

## 3. Entender la estructura del dataset

Cada fila es una lectura de sensores en un ciclo concreto de un motor concreto.
Un 'ciclo' equivale aproximadamente a un vuelo completo.

In [ ]:
# ¿Cuántos motores distintos hay en el dataset?
n_motores = df['motor_id'].nunique()
print(f'Número de motores: {n_motores}')

# ¿Cuántos ciclos tiene cada motor de media?
ciclos_por_motor = df.groupby('motor_id')['ciclo'].max()
print(f'\nCiclos hasta el fallo (vida útil):')
print(f'  Mínimo: {ciclos_por_motor.min()} ciclos')
print(f'  Máximo: {ciclos_por_motor.max()} ciclos')
print(f'  Media:  {ciclos_por_motor.mean():.0f} ciclos')

## 4. Calcular el RUL — Remaining Useful Life

El dataset no tiene directamente la columna que queremos predecir.
Necesitamos calcular el **RUL (Vida Útil Restante)** de cada motor en cada ciclo.

**Fórmula:** RUL = ciclo_máximo_del_motor - ciclo_actual

Ejemplo: si un motor dura 200 ciclos y estamos en el ciclo 150, RUL = 50.

In [ ]:
# Calculamos el ciclo máximo de cada motor (cuando falló)
ciclo_max = df.groupby('motor_id')['ciclo'].max().reset_index()
ciclo_max.columns = ['motor_id', 'ciclo_max']

# Unimos con el dataframe principal
df = df.merge(ciclo_max, on='motor_id')

# Calculamos el RUL
df['RUL'] = df['ciclo_max'] - df['ciclo']

# Ya no necesitamos ciclo_max
df = df.drop('ciclo_max', axis=1)

print('RUL calculado ✅')
print(f'Rango de RUL: {df["RUL"].min()} — {df["RUL"].max()} ciclos')
df[['motor_id', 'ciclo', 'RUL']].head(10)

## 5. Crear la variable objetivo (TARGET)

Convertimos el problema de regresión (predecir RUL exacto) en **clasificación binaria**:

- **TARGET = 1** → el motor fallará en los próximos 30 ciclos (EN RIESGO)
- **TARGET = 0** → el motor tiene más de 30 ciclos de vida restante (SEGURO)

¿Por qué 30 ciclos? Es el tiempo que necesita un equipo de mantenimiento para
planificar y ejecutar la revisión del motor. Este umbral se puede ajustar.

In [ ]:
# Umbral de riesgo: 30 ciclos
UMBRAL_RIESGO = 30

# Creamos el target: 1 si en riesgo, 0 si seguro
df['target'] = (df['RUL'] < UMBRAL_RIESGO).astype(int)

# ¿Cuántos ciclos están en riesgo vs seguros?
distribucion = df['target'].value_counts()
porcentaje = df['target'].value_counts(normalize=True) * 100

print('Distribución del target:')
print(f'  Seguro  (0): {distribucion[0]:,} ciclos ({porcentaje[0]:.1f}%)')
print(f'  En riesgo (1): {distribucion[1]:,} ciclos ({porcentaje[1]:.1f}%)')
print(f'\nEl dataset está desbalanceado: hay más ciclos "seguros" que "en riesgo"')
print(f'Esto lo tendremos en cuenta al entrenar el modelo.')

## 6. Gráfico 1 — Distribución de la vida útil de los motores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ciclos_por_motor.hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(ciclos_por_motor.mean(), color='red', linestyle='--', 
           label=f'Media: {ciclos_por_motor.mean():.0f} ciclos')
ax.set_xlabel('Ciclos hasta el fallo (vida útil)')
ax.set_ylabel('Número de motores')
ax.set_title('Distribución de la vida útil de los motores')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig1_vida_util.png', dpi=150)
plt.show()

print('\n📌 Conclusión: La vida útil varía bastante entre motores.')
print('Esto refleja variaciones en condiciones de uso y fabricación.')

## 7. Gráfico 2 — Degradación de sensores a lo largo del tiempo

Visualizamos cómo cambian algunos sensores conforme el motor se acerca al fallo.

In [ ]:
# Tomamos un motor de ejemplo para visualizar su degradación
motor_ejemplo = df[df['motor_id'] == 1].copy()

# Sensores que suelen mostrar degradación clara
sensores_interesantes = ['s2', 's3', 's4', 's11', 's12', 's15']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, sensor in enumerate(sensores_interesantes):
    axes[i].plot(motor_ejemplo['ciclo'], motor_ejemplo[sensor], 
                 color='steelblue', linewidth=1)
    axes[i].axvline(motor_ejemplo['ciclo'].max() - 30, 
                    color='red', linestyle='--', alpha=0.7,
                    label='Zona de riesgo')
    axes[i].set_title(f'Sensor {sensor}')
    axes[i].set_xlabel('Ciclo')
    axes[i].legend(fontsize=8)

plt.suptitle('Degradación de sensores — Motor 1', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig2_degradacion_sensores.png', dpi=150)
plt.show()

print('\n📌 Conclusión: Algunos sensores muestran una tendencia clara')
print('de degradación antes del fallo. Esos serán los más útiles para el modelo.')

## 8. Gráfico 3 — Sensores con varianza cero (inútiles)

Algunos sensores no cambian nunca — siempre tienen el mismo valor.
Esos sensores no aportan información al modelo y los eliminaremos.

In [ ]:
# Columnas de sensores
cols_sensores = [c for c in df.columns if c.startswith('s')]

# Calculamos la varianza de cada sensor
varianzas = df[cols_sensores].var().sort_values()

# Sensores con varianza casi cero (umbral muy pequeño)
sensores_inutiles = varianzas[varianzas < 0.001].index.tolist()

fig, ax = plt.subplots(figsize=(12, 4))
colores = ['red' if v < 0.001 else 'steelblue' for v in varianzas]
ax.bar(varianzas.index, varianzas.values, color=colores)
ax.set_xlabel('Sensor')
ax.set_ylabel('Varianza')
ax.set_title('Varianza por sensor (rojo = sin información útil)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/fig3_varianza_sensores.png', dpi=150)
plt.show()

print(f'\n📌 Sensores sin información útil: {sensores_inutiles}')
print('Los eliminaremos en el notebook de preprocesamiento.')

## 9. Gráfico 4 — Correlación de sensores con el RUL

¿Qué sensores tienen más relación con la vida útil restante?
Los sensores con mayor correlación (positiva o negativa) serán los más importantes.

In [ ]:
# Correlación de cada sensor con el RUL
correlaciones = df[cols_sensores + ['RUL']].corr()['RUL'].drop('RUL')
correlaciones = correlaciones.sort_values()

fig, ax = plt.subplots(figsize=(12, 5))
colores = ['red' if c < 0 else 'green' for c in correlaciones]
ax.barh(correlaciones.index, correlaciones.values, color=colores, alpha=0.7)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlación con RUL')
ax.set_title('Correlación de sensores con la Vida Útil Restante (RUL)\nRojo = sube cuando el motor se acerca al fallo | Verde = baja cuando se acerca al fallo')
plt.tight_layout()
plt.savefig('../data/processed/fig4_correlacion_rul.png', dpi=150)
plt.show()

# Top 5 sensores más correlacionados
top5 = correlaciones.abs().sort_values(ascending=False).head(5)
print(f'\n📌 Top 5 sensores más relacionados con el fallo:')
for sensor, corr in top5.items():
    print(f'  {sensor}: correlación = {correlaciones[sensor]:.3f}')

## 10. Resumen del EDA

Aquí resumimos todo lo que hemos aprendido en este notebook.

In [ ]:
print('=' * 55)
print('RESUMEN DEL EDA')
print('=' * 55)
print(f'Motores en el dataset:        {n_motores}')
print(f'Total de registros:           {len(df):,}')
print(f'Vida útil media:              {ciclos_por_motor.mean():.0f} ciclos')
print(f'Ciclos en riesgo (RUL < 30):  {df["target"].sum():,} ({df["target"].mean()*100:.1f}%)')
print(f'Sensores inútiles detectados: {len(sensores_inutiles)}')
print(f'Sensores útiles restantes:    {len(cols_sensores) - len(sensores_inutiles)}')
print()
print('PRÓXIMOS PASOS (notebook 02):')
print('  1. Eliminar sensores con varianza cero')
print('  2. Normalizar lecturas de sensores')
print('  3. Crear features de medias móviles')
print('  4. Preparar datos para el modelo')
print('=' * 55)

---
**Notebook completado ✅**  
Siguiente paso: `02_preprocesamiento.ipynb`